# Simulation PSF & mask pupillaire

In [1]:
# Rend le paquet importable sans installation préalable
# (pip install -e . depuis la racine du dépôt fonctionne aussi).
import sys
from pathlib import Path as _Path
sys.path.insert(0, str((_Path.cwd() / ".." / "src").resolve()))

import numpy as np
import imageio.v3 as iio
import poppy.zernike as zern
from plotly.subplots import make_subplots

from eoqual.phaseretrieval.io_utils import plot_array_plotly


# --------------------------------------------------
# Paramètres
# --------------------------------------------------

N = 256
WAVELENGTH = 550e-9  # m

DEFOCUS_WAVES = 0.1
ASTIG_WAVES   = 0.05
DIVERSITY_DEFOCUS_WAVES = 0.5
print("I - Aberrations injectées:")
print(f"   > Defocus       : {DEFOCUS_WAVES:.3f} λ")
print(f"   > Astigmatisme  : {ASTIG_WAVES:.3f} λ")
print("Id - Aberrations injectées:")
print(f"   > Defocus       : {DEFOCUS_WAVES + DIVERSITY_DEFOCUS_WAVES:.3f} λ")
print(f"   > Astigmatisme  : {ASTIG_WAVES:.3f} λ")

# conversion λ -> radians
DEFOCUS_RAD = 2 * np.pi * DEFOCUS_WAVES
ASTIG_RAD   = 2 * np.pi * ASTIG_WAVES
DIVERSITY_DEFOCUS_RAD = 2 * np.pi * DIVERSITY_DEFOCUS_WAVES

# --------------------------------------------------
# Masque circulaire
# --------------------------------------------------

y, x = np.indices((N, N))

cx = N // 2
cy = N // 2

radius = int(0.4 * N)
r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)

mask = (r <= radius).astype(np.uint8)
print(f"Mask sum : {int(mask.sum())}")

# --------------------------------------------------
# Base de Zernike
# --------------------------------------------------
basis = zern.zernike_basis(
    nterms=36,
    npix=N,
    outside=0.0,
)

# Noll :
# 1 piston
# 2 tilt X
# 3 tilt Y
# 4 defocus
# 5 astig 45°
# 6 astig 0°

# --------------------------------------------------
# I - Phase
# --------------------------------------------------
phase = np.zeros((N, N))
# 0.1 λ de défocus
phase += DEFOCUS_RAD * basis[3]
# 0.05 λ d'astigmatisme
phase += ASTIG_RAD * basis[4]
phase *= mask

z4_true_nm = DEFOCUS_WAVES * WAVELENGTH * 1e9
z5_true_nm = ASTIG_WAVES   * WAVELENGTH * 1e9
print("I - Valeurs théoriques injectées:")
print(f"   > Z4 (Defocus)      : {z4_true_nm:.3f} nm")
print(f"   > Z5 (Astigmatisme) : {z5_true_nm:.3f} nm")

# --------------------------------------------------
# Id - Phase diversity
# --------------------------------------------------
phase_diversity = phase.copy()
# Z4 supplémentaire connu
phase_diversity += (
    DIVERSITY_DEFOCUS_RAD
    * basis[3]
)
phase_diversity *= mask

z4_true_nm = (DEFOCUS_WAVES + DIVERSITY_DEFOCUS_WAVES) * WAVELENGTH * 1e9
z5_true_nm = ASTIG_WAVES   * WAVELENGTH * 1e9
print("Id * Valeurs théoriques injectées:")
print(f"   > Z4 (Defocus)      : {z4_true_nm:.3f} nm")
print(f"   > Z5 (Astigmatisme) : {z5_true_nm:.3f} nm")

# --------------------------------------------------
# I - Calcul PSF diversity
# --------------------------------------------------
pupil = mask * np.exp(1j * phase)
focal = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(pupil)))
psf  = np.abs(focal) ** 2
psf /= psf.sum()

# --------------------------------------------------
# Id - Calcul PSF 
# --------------------------------------------------
pupil_div = mask * np.exp(1j * phase_diversity)
focal_div = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(pupil_div)))
psf_div = np.abs(focal_div)**2
psf_div /= psf_div.sum()

# --------------------------------------------------
# Sauvegarde TIFF
# --------------------------------------------------
print("Save datas:")
iio.imwrite(
    "data/mask_simu.tiff",
    mask.astype(np.uint8)
)
print("   > data/mask_simu.tiff écrit")

iio.imwrite(
    "data/psf_simu.tiff",
    psf.astype(np.float32)
)
print("   > data/psf_simu.tiff écrit")

iio.imwrite(
    "data/phase_simu.tiff",
    phase.astype(np.float32)
)
print("   > data/phase_simu.tiff écrit")

iio.imwrite(
    "data/psf_diversity_simu.tiff",
    psf_div.astype(np.float32)
)
print("   > data/psf_diversity_simu.tiff écrit")

# --------------------------------------------------
# Visualisation
# --------------------------------------------------
fig = make_subplots(rows=1, cols=2, subplot_titles=("Focused PSF (log)", "Defocused PSF (log)"))
for col, arr in enumerate((psf, psf_div), start=1):
    single = plot_array_plotly(arr, log=True)
    fig.add_trace(single.data[0], row=1, col=col)
fig.update_layout(width=900, height=450, margin=dict(l=0, r=0, t=40, b=0))
fig.update_yaxes(autorange="reversed")
fig.show()

#EOF

I - Aberrations injectées:
   > Defocus       : 0.100 λ
   > Astigmatisme  : 0.050 λ
Id - Aberrations injectées:
   > Defocus       : 0.600 λ
   > Astigmatisme  : 0.050 λ
Mask sum : 32681
I - Valeurs théoriques injectées:
   > Z4 (Defocus)      : 55.000 nm
   > Z5 (Astigmatisme) : 27.500 nm
Id * Valeurs théoriques injectées:
   > Z4 (Defocus)      : 330.000 nm
   > Z5 (Astigmatisme) : 27.500 nm
Save datas:
   > data/mask_simu.tiff écrit
   > data/psf_simu.tiff écrit
   > data/phase_simu.tiff écrit
   > data/psf_diversity_simu.tiff écrit


## Simulation Bayer (R / Gu / Gl / B) avec décalage chromatique

Réutilise le masque, la base de Zernike et les aberrations (`DEFOCUS_WAVES`,
`ASTIG_WAVES`) définis ci-dessus pour produire, **en plus** de la PSF
monochromatique, une image brute au format Bayer (motif `RGGB` — même
convention que `starpsfsr.telescope.BayerExtractor` : `(0,0)=R`, `(0,1)=Gu`,
`(1,0)=Gl`, `(1,1)=B`), avec un **décalage chromatique connu** entre bandes.

Modèle physique retenu :
- La WFE (en nm) est une propriété **géométrique** de l'optique,
  indépendante de la longueur d'onde ; sa conversion en radians
  (`2π·WFE_nm/λ_nm`) dépend en revanche de la bande — la même aberration
  physique produit donc une phase différente en R, G et B.
- Un décalage latéral **connu et contrôlable** (`CHROMA_SHIFT_PX`, en
  pixels détecteur) est injecté par bande, pour simuler une aberration
  chromatique latérale (couleur transverse) — c'est ce décalage que la
  correction évoquée pour `method_epsf_bayer.py`/`shift_tools_bayer.py`
  (diagnostic dans `REFERENCE_TECHNIQUE.md`) devra récupérer.
- La **taille** de la tache de diffraction dépend elle aussi de λ
  (∝ λ/D à échelle pixel détecteur fixe) : `optics.psf_from_phase` (une
  simple FFT sur une grille fixe) ne le modélise pas — le rapport λ/N
  s'annule dans le calcul, la taille en pixels ne dépend alors plus de λ.
  On utilise ici `simu_star_profiles.psf_from_pupil_field_wavelength_scaled`,
  qui rééchantillonne la pupille (zero-padding pour le rouge, recadrage
  pour le bleu) avant la FFT pour restaurer cette dépendance physique,
  vérifiée numériquement (`peak × (λ/λ_G)² ≈ cst` sur les 3 bandes).

**Convention de signe** : `CHROMA_SHIFT_PX[band]` est le décalage *appliqué*
à la bande (translation du contenu). Si on mesure ensuite ce décalage avec
`skimage.registration.phase_cross_correlation(reference_image=psf_G,
moving_image=psf_R)` — exactement l'appel fait par
`starpsfsr.star_PSF_SR_bayer.estimate_psf_channel_offsets` — le résultat
est l'**opposé** (le décalage *à appliquer à R pour le recaler sur G*), par
construction de cette fonction. Vérifié numériquement : décalage R injecté
`(+0.60, -0.30)` → mesuré `(-0.64, +0.26)`.

In [2]:
import json

from eoqual.phaseretrieval.simulation.star_profiles import psf_from_pupil_field_wavelength_scaled

# --------------------------------------------------
# II - Paramètres Bayer
# --------------------------------------------------

# Longueurs d'onde centrales par bande (nm). G reprend la référence
# utilisée pour la PSF monochromatique ci-dessus (WAVELENGTH = 550 nm).
WAVELENGTHS_NM = {
    "R": 650.0,
    "G": WAVELENGTH * 1e9,
    "B": 450.0,
}

# Décalage chromatique connu (dy, dx), en pixels détecteur, relatif à G
CHROMA_SHIFT_PX = {
    "R": (0.6, -0.3),
    "G": (0.0, 0.0),
    "B": (-0.4, 0.5),
}

# WFE physique (nm), indépendante de la bande -- seule sa conversion
# en radians dépend de lambda (voir note de modèle physique ci-dessus)
DEFOCUS_NM = DEFOCUS_WAVES * (WAVELENGTH * 1e9)
ASTIG_NM   = ASTIG_WAVES   * (WAVELENGTH * 1e9)

print("II - Simulation Bayer:")
print(f"   > WFE Defocus (physique)      : {DEFOCUS_NM:.3f} nm")
print(f"   > WFE Astigmatisme (physique) : {ASTIG_NM:.3f} nm")

II - Simulation Bayer:
   > WFE Defocus (physique)      : 55.000 nm
   > WFE Astigmatisme (physique) : 27.500 nm


In [3]:
def fourier_shift(img: np.ndarray, dy: float, dx: float) -> np.ndarray:
    """
    Décale une image de (dy, dx) pixels par rampe de phase (sub-pixel, FFT).

    Parameters
    ----------
    img : np.ndarray
        Image source (2-D), énergie positive.
    dy, dx : float
        Décalage vertical / horizontal, en pixels (positif = vers le
        bas / la droite).

    Returns
    -------
    np.ndarray
        Image décalée, renormalisée (énergie = 1).
    """
    h, w = img.shape
    fft_img = np.fft.fft2(img)
    u = np.fft.fftfreq(h).reshape(-1, 1)
    v = np.fft.fftfreq(w).reshape(1, -1)
    phase_ramp = np.exp(-2j * np.pi * (dy * u + dx * v))
    shifted = np.real(np.fft.ifft2(fft_img * phase_ramp))
    shifted = np.maximum(shifted, 0)
    return shifted / shifted.sum()


# --------------------------------------------------
# II - PSF par bande (phase dépendante de lambda + décalage chromatique)
# --------------------------------------------------
psf_band   = {}
phase_band = {}
for band, wl_nm in WAVELENGTHS_NM.items():
    phase_c  = (DEFOCUS_NM * basis[3] + ASTIG_NM * basis[4]) * (2 * np.pi / wl_nm)
    phase_c *= mask

    pupil_c = mask * np.exp(1j * phase_c)
    wavelength_ratio = wl_nm / WAVELENGTHS_NM["G"]
    psf_c = psf_from_pupil_field_wavelength_scaled(pupil_c, wavelength_ratio)

    dy, dx = CHROMA_SHIFT_PX[band]
    psf_band[band]   = fourier_shift(psf_c, dy, dx)
    phase_band[band] = phase_c

    print(f"   > Bande {band} (λ={wl_nm:.0f} nm) : "
          f"Z4={DEFOCUS_NM:.2f} nm, Z5={ASTIG_NM:.2f} nm, "
          f"décalage=({dy:+.2f}, {dx:+.2f}) px")

   > Bande R (λ=650 nm) : Z4=55.00 nm, Z5=27.50 nm, décalage=(+0.60, -0.30) px
   > Bande G (λ=550 nm) : Z4=55.00 nm, Z5=27.50 nm, décalage=(+0.00, +0.00) px
   > Bande B (λ=450 nm) : Z4=55.00 nm, Z5=27.50 nm, décalage=(-0.40, +0.50) px


In [4]:
# --------------------------------------------------
# Mosaïque Bayer RGGB
# (0,0)=R  (0,1)=Gu  (1,0)=Gl  (1,1)=B -- convention BayerExtractor (starpsfsr)
# --------------------------------------------------
bayer = np.zeros((N, N), dtype=np.float64)
bayer[0::2, 0::2] = psf_band["R"][0::2, 0::2]
bayer[0::2, 1::2] = psf_band["G"][0::2, 1::2]   # Gu
bayer[1::2, 0::2] = psf_band["G"][1::2, 0::2]   # Gl
bayer[1::2, 1::2] = psf_band["B"][1::2, 1::2]

# --------------------------------------------------
# Sauvegarde
# --------------------------------------------------
print("Save datas Bayer:")
for band in WAVELENGTHS_NM:
    iio.imwrite(f"data/psf_simu_{band}.tiff", psf_band[band].astype(np.float32))
    print(f"   > data/psf_simu_{band}.tiff écrit")

iio.imwrite("data/psf_simu_bayer.tiff", bayer.astype(np.float32))
print("   > data/psf_simu_bayer.tiff écrit (mosaïque RGGB, pleine résolution)")

truth = {
    "bayer_pattern": "RGGB",
    "wavelengths_nm": WAVELENGTHS_NM,
    "chroma_shift_px_dy_dx": CHROMA_SHIFT_PX,
    "wfe_nm": {"Z4": DEFOCUS_NM, "Z5": ASTIG_NM},
}
with open("data/psf_simu_bayer_truth.json", "w") as f:
    json.dump(truth, f, indent=2)
print("   > data/psf_simu_bayer_truth.json écrit (vérité terrain)")

Save datas Bayer:
   > data/psf_simu_R.tiff écrit
   > data/psf_simu_G.tiff écrit
   > data/psf_simu_B.tiff écrit
   > data/psf_simu_bayer.tiff écrit (mosaïque RGGB, pleine résolution)
   > data/psf_simu_bayer_truth.json écrit (vérité terrain)


In [5]:
# --------------------------------------------------
# Visualisation
# --------------------------------------------------
fig = make_subplots(rows=1, cols=4, subplot_titles=("R", "G", "B", "Bayer RGGB (brut)"))
for col, arr in enumerate((psf_band["R"], psf_band["G"], psf_band["B"], bayer), start=1):
    single = plot_array_plotly(arr, log=True)
    fig.add_trace(single.data[0], row=1, col=col)
fig.update_layout(width=1300, height=380, margin=dict(l=0, r=0, t=40, b=0))
fig.update_yaxes(autorange="reversed")
fig.show()

#EOF